In [ ]:
import requests
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = StringIO(requests.get('https://raw.githubusercontent.com/rajtilakls2510/car_price_predictor/refs/heads/master/quikr_car.csv').text)
df = pd.read_csv(df)
backdup = df.copy()

In [ ]:
df.sample(4)

## PROBLEMS IN DATASET
- convert categorical columns into lowercase.
- filter out noise from company.
- filter out noise from year.
- remove 'Ask For Price' from Price.
- remove values of kms_driven with no kms at end ('Petrol') and remove kms from end.
- remove noise from name column

In [ ]:
df.dtypes

In [ ]:
df['company'].unique()

In [ ]:
valid_brands = [
    'hyundai','mahindra','maruti','ford','skoda','audi','toyota',
    'renault','honda','datsun','mitsubishi','tata','volkswagen',
    'chevrolet','mini','bmw','nissan','hindustan','fiat','force',
    'mercedes','jaguar','jeep','volvo','land rover'
]

In [ ]:
df['year'].unique()

In [ ]:
len(df[df['Price'] == 'Ask For Price'])

In [ ]:
df['kms_driven'].value_counts()

In [ ]:
df['fuel_type'].value_counts()

In [ ]:
df['name'].value_counts()

In [ ]:
def make_lower(column):
    df[column] = df[column].str.lower().str.strip()

In [ ]:
make_lower('company')
df = df[df['company'].isin(valid_brands)]
df['company']

In [ ]:
df = df[df['year'].str[1] == '0']
df['year'] = df['year'].astype(int)

In [ ]:
df.shape

In [ ]:
df = df[df['Price'] != 'Ask For Price']

In [ ]:
df['Price'] = df['Price'].str.strip(' ').str.replace(',', '').astype(int)
df.shape

In [ ]:
df = df[df['kms_driven'].str.contains("kms")]
df.shape

In [ ]:
df['kms_driven'] = df['kms_driven'].str.replace(',', '').str.strip('kms').astype(int)
df['kms_driven'].unique()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(keep='first', inplace=True)

In [ ]:
df.shape

In [ ]:
df['name'] = df['name'].str.lower().str.strip(' ')

In [ ]:
df['name'].unique()

In [ ]:
df.sample(5)

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
plt.hist(df['Price'], bins=20)
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
len(df[df['Price'] > df['Price'].quantile(0.95)])

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(df['Price'], vert=False)
plt.title('Price Distribution')
plt.xlabel('Price')
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
df.boxplot(column='Price', by='company', rot=90)
plt.title('Price by Company')
plt.suptitle('')
plt.show()

In [ ]:
q1 = df['Price'].quantile(0.25)
q3 = df['Price'].quantile(0.75)
iqr = q3 - q1
lowerbound = q1 - 1.5 * iqr
upperbound = q3 + 1.5 * iqr

In [ ]:
df['Price'] = df['Price'].clip(lowerbound, upperbound)
df.dtypes

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe = OneHotEncoder()
ohe.fit(df[['name', 'company', 'fuel_type']])

In [ ]:
ohe.categories_